In [22]:
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os
import warnings
import json

# Suppress warnings
warnings.simplefilter(action='ignore', category=Warning)

fsize = 15
plt.rcParams.update({"font.size": fsize})
%config InlineBackend.figure_format = 'retina'

In [23]:
def load_evidence(fn, ds_name):
    # we only want derived since we're going to compare gene ids
    df = pd.read_json(fn)
    ddf = df['derived'].apply(pd.Series)
    ddf["ds_name"] = ds_name

    # normalize the string values for cell types
    ddf["cell_type_label"] = ddf["cell_type_label"].str.strip().str.upper()
    ddf["cell_type_id"] = ddf["cell_type_id"].str.strip().str.upper()
    
    # normalize the string values for genes
    ddf["feature_name"] = ddf["feature_name"].str.strip().str.upper()
    ddf["feature_identifier"] = ddf["feature_identifier"].str.strip().str.upper()
    hddf = ddf.query("organism == 'homo_sapiens'")
    return hddf

def get_ctmap(ctmap_fn):
    with open(ctmap_fn, 'r') as file:
        map_dict = json.load(file)
        upper_map = {k.upper(): [v.upper() for v in vals] for k, vals in map_dict.items()}
        rev_map = {v: k for k, vals in upper_map.items() for v in vals}
    return (upper_map, rev_map)

In [6]:
fns = glob.glob("../../data/*")

In [122]:
hmns = []
degs = []
llms = []

gmap_fn = "../../data/gmap.txt"
gmap = (pd.read_csv(gmap_fn, header=None, sep="\t", names=["gname", "ensembl_id"])
    .drop_duplicates("gname")
    .applymap(lambda x: x.strip().upper() if isinstance(x, str) else x)
).set_index("gname")["ensembl_id"].to_dict()

for fn in fns:
    ds_name = fn.split("/")[-1]
    hmn_fn = os.path.join(fn, "evidence_human/evidence.json")
    deg_fn = os.path.join(fn, "evidence_deg/evidence.json")
    llm_fn = glob.glob(os.path.join(fn, "evidence_llm*/evidence.json"))
    llm_fn = llm_fn[0] if len(llm_fn) > 0 else None
    ctmap_fn = os.path.join(fn, "ctmap/ctmap.json")


    if os.path.exists(ctmap_fn):
        ctmap, rev_ctmap = get_ctmap(ctmap_fn)
        if os.path.exists(hmn_fn):
            hmn = load_evidence(hmn_fn, ds_name)
            hmn["cell_type_id_mapped"] = hmn["cell_type_label"].map(lambda x: rev_ctmap.get(x.strip().upper(), None))
            hmn["found_cell_type_id"] = hmn["cell_type_id"] == hmn["cell_type_id_mapped"]

            hmn["feature_id_mapped"] = hmn["feature_name"].map(lambda x: gmap.get(x.strip().upper(), None))
            hmn["found_feature_id"] = hmn["feature_identifier"] == hmn["feature_id_mapped"]

            hmns.append(hmn)

        if os.path.exists(deg_fn):
            deg = load_evidence(deg_fn, ds_name)
            deg["cell_type_id_mapped"] = deg["cell_type_label"].map(lambda x: rev_ctmap.get(x.strip().upper(), None))
            deg["found_cell_type_id"] = deg["cell_type_id"] == deg["cell_type_id_mapped"]

            deg["feature_id_mapped"] = deg["feature_name"].map(lambda x: gmap.get(x.strip().upper(), None))
            deg["found_feature_id"] = deg["feature_identifier"] == deg["feature_id_mapped"]

            degs.append(deg)
        if os.path.exists(llm_fn):
            llm = load_evidence(llm_fn, ds_name)
            llm["cell_type_id_mapped"] = llm["cell_type_label"].map(lambda x: rev_ctmap.get(x.strip().upper(), None))
            llm["found_cell_type_id"] = llm["cell_type_id"] == llm["cell_type_id_mapped"]

            llm["feature_id_mapped"] = llm["feature_name"].map(lambda x: gmap.get(x.strip().upper(), None))
            llm["found_feature_id"] = llm["feature_identifier"] == llm["feature_id_mapped"]

            llms.append(llm)

hmn = pd.concat(hmns)
deg = pd.concat(degs)
llm = pd.concat(llms)

# Human annotations

In [123]:
agg = hmn.groupby("ds_name").agg({
    "found_feature_id": ["count", "sum"], 
    "found_cell_type_id": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_feature_id", "pct")] = 100*agg[("found_feature_id", "found")] / agg[("found_feature_id", "total")]
agg[("found_cell_type_id", "pct")] = 100*agg[("found_cell_type_id", "found")] / agg[("found_cell_type_id", "total")]
agg[("found_feature_id", "complete")] = agg[("found_feature_id", "pct")] == 100
agg[("found_cell_type_id", "complete")] = agg[("found_cell_type_id", "pct")] == 100
agg["all_complete"] = agg[("found_feature_id", "complete")] & agg[("found_cell_type_id", "complete")]

In [124]:
# Add a total row at the bottom of the DataFrame
total_row = agg.sum(numeric_only=True)
total_row[("found_feature_id", "pct")] = 100 * total_row[("found_feature_id", "found")] / total_row[("found_feature_id", "total")]
total_row[("found_cell_type_id", "pct")] = 100 * total_row[("found_cell_type_id", "found")] / total_row[("found_cell_type_id", "total")]

# Handling boolean columns for completeness
total_row[("found_feature_id", "complete")] = total_row[("found_feature_id", "pct")] == 100
total_row[("found_cell_type_id", "complete")] = total_row[("found_cell_type_id", "pct")] == 100
total_row["all_complete"] = total_row[("found_feature_id", "complete")] & total_row[("found_cell_type_id", "complete")]

# Add the total row to the DataFrame
agg.loc["total"] = total_row

# Sort the columns
agg = agg.sort_index(axis=1)

agg.reset_index()

ds_name all_complete found_cell_type_id                      \
                                                complete   found         pct   
0      adipose_Emont2022        False               True   373.0  100.000000   
1   adipose_Hildreth2021        False              False   162.0   95.294118   
2     adipose_Jaitin2019         True               True    11.0  100.000000   
3    adipose_Massier2023        False               True   258.0  100.000000   
4    adipose_Merrick2019        False              False    28.0   96.551724   
5      adipose_Vijay2019        False               True   105.0  100.000000   
6         bladder_Yu2019        False               True    79.0  100.000000   
7            bone_He2021        False              False   398.0   99.749373   
8         eye_Gautam2021        False               True   373.0  100.000000   
9       heart_Tucker2020        False               True   559.0  100.000000   
10                 total        False              False  2346.0   99.575552   

           found_feature_id                              
     total         complete   found         pct   total  
0    373.0            False   367.0   98.391421   373.0  
1    170.0            False   160.0   94.117647   170.0  
2     11.0             True    11.0  100.000000    11.0  
3    258.0            False   251.0   97.286822   258.0  
4     29.0            False    21.0   72.413793    29.0  
5    105.0            False   103.0   98.095238   105.0  
6     79.0            False    75.0   94.936709    79.0  
7    399.0            False   385.0   96.491228   399.0  
8    373.0            False   352.0   94.369973   373.0  
9    559.0            False   531.0   94.991055   559.0  
10  2356.0            False  2256.0   95.755518  2356.0

In [140]:
# find specific datasets with missing cell type or feature name
ds = 'bone_He2021'
hmn.query(
    f"ds_name == '{ds}' & (~found_cell_type_id | ~found_feature_id)"
)[["cell_type_label", "cell_type_id",  "cell_type_id_mapped", "feature_name", "feature_identifier",  "feature_id_mapped"]].head(3)

,cell_type_label,cell_type_id,cell_type_id_mapped,feature_name,feature_identifier,feature_id_mapped
22,PERICHONDRIAL MESENCHYMAL STROMAL CELL 6,PERICHONDRIAL MESENCHYMAL STROMAL CELL,PERICHONDRIAL MESENCHYMAL STROMAL CELL,NOV,ENSG00000147403,ENSG00000136999
27,PROXIMAL MESENCHYMAL CELL - LBM3,MESOTHELIUM,MESOTHELIUM,HOX2-6,ENSG00000172789,None
28,DISTAL MESENCHYMAL CELL - LBM1,LIMB BUD MESENCHYMAL CELL,LIMB BUD MESENCHYMAL CELL,HOX9-11,ENSG00000123407,None


# DEG annotations

In [126]:
agg = deg.groupby("ds_name").agg({
    "found_feature_id": ["count", "sum"], 
    "found_cell_type_id": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_feature_id", "pct")] = 100*agg[("found_feature_id", "found")] / agg[("found_feature_id", "total")]
agg[("found_cell_type_id", "pct")] = 100*agg[("found_cell_type_id", "found")] / agg[("found_cell_type_id", "total")]
agg[("found_feature_id", "complete")] = agg[("found_feature_id", "pct")] == 100
agg[("found_cell_type_id", "complete")] = agg[("found_cell_type_id", "pct")] == 100
agg["all_complete"] = agg[("found_feature_id", "complete")] & agg[("found_cell_type_id", "complete")]

In [127]:
# Add a total row at the bottom of the DataFrame
total_row = agg.sum(numeric_only=True)
total_row[("found_feature_id", "pct")] = 100 * total_row[("found_feature_id", "found")] / total_row[("found_feature_id", "total")]
total_row[("found_cell_type_id", "pct")] = 100 * total_row[("found_cell_type_id", "found")] / total_row[("found_cell_type_id", "total")]

# Handling boolean columns for completeness
total_row[("found_feature_id", "complete")] = total_row[("found_feature_id", "pct")] == 100
total_row[("found_cell_type_id", "complete")] = total_row[("found_cell_type_id", "pct")] == 100
total_row["all_complete"] = total_row[("found_feature_id", "complete")] & total_row[("found_cell_type_id", "complete")]

# Add the total row to the DataFrame
agg.loc["total"] = total_row

# Sort the columns
agg = agg.sort_index(axis=1)

agg.reset_index()

ds_name all_complete found_cell_type_id                      \
                                               complete   found         pct   
0     adipose_Emont2022        False               True   347.0  100.000000   
1  adipose_Hildreth2021        False              False   200.0   10.526316   
2    adipose_Jaitin2019        False               True   160.0  100.000000   
3   adipose_Massier2023        False               True    62.0  100.000000   
4     adipose_Vijay2019        False               True    53.0  100.000000   
5        bladder_Yu2019        False               True    67.0  100.000000   
6           bone_He2021        False               True   372.0  100.000000   
7        eye_Gautam2021        False               True   420.0  100.000000   
8      heart_Tucker2020        False               True   258.0  100.000000   
9                 total        False              False  1939.0   53.283869   

          found_feature_id                             
    total         complete   found        pct   total  
0   347.0            False   314.0  90.489914   347.0  
1  1900.0            False  1839.0  96.789474  1900.0  
2   160.0            False   158.0  98.750000   160.0  
3    62.0            False    61.0  98.387097    62.0  
4    53.0            False    49.0  92.452830    53.0  
5    67.0            False    64.0  95.522388    67.0  
6   372.0            False   356.0  95.698925   372.0  
7   420.0            False   396.0  94.285714   420.0  
8   258.0            False   250.0  96.899225   258.0  
9  3639.0            False  3487.0  95.823028  3639.0

In [139]:
# find specific datasets with missing cell type or feature name
ds = 'bone_He2021'
deg.query(
    f"ds_name == '{ds}' & (~found_cell_type_id | ~found_feature_id)"
)[["cell_type_label", "cell_type_id",  "cell_type_id_mapped", "feature_name", "feature_identifier",  "feature_id_mapped"]].head(3)

,cell_type_label,cell_type_id,cell_type_id_mapped,feature_name,feature_identifier,feature_id_mapped
17,OCP,OSTEOCHONDROGENIC PROGENITOR,OSTEOCHONDROGENIC PROGENITOR,OSR1,ENSG00000172939,ENSG00000143867
68,MAROPHAGE,MACROPHAGE,MACROPHAGE,ELF1,ENSG00000130165,ENSG00000120690
80,MAROPHAGE,MACROPHAGE,MACROPHAGE,LAT2,ENSG00000092068,ENSG00000086730


# LLM annotations

In [128]:
agg = hmn.groupby("ds_name").agg({
    "found_feature_id": ["count", "sum"], 
    "found_cell_type_id": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_feature_id", "pct")] = 100*agg[("found_feature_id", "found")] / agg[("found_feature_id", "total")]
agg[("found_cell_type_id", "pct")] = 100*agg[("found_cell_type_id", "found")] / agg[("found_cell_type_id", "total")]
agg[("found_feature_id", "complete")] = agg[("found_feature_id", "pct")] == 100
agg[("found_cell_type_id", "complete")] = agg[("found_cell_type_id", "pct")] == 100
agg["all_complete"] = agg[("found_feature_id", "complete")] & agg[("found_cell_type_id", "complete")]

In [129]:
# Add a total row at the bottom of the DataFrame
total_row = agg.sum(numeric_only=True)
total_row[("found_feature_id", "pct")] = 100 * total_row[("found_feature_id", "found")] / total_row[("found_feature_id", "total")]
total_row[("found_cell_type_id", "pct")] = 100 * total_row[("found_cell_type_id", "found")] / total_row[("found_cell_type_id", "total")]

# Handling boolean columns for completeness
total_row[("found_feature_id", "complete")] = total_row[("found_feature_id", "pct")] == 100
total_row[("found_cell_type_id", "complete")] = total_row[("found_cell_type_id", "pct")] == 100
total_row["all_complete"] = total_row[("found_feature_id", "complete")] & total_row[("found_cell_type_id", "complete")]

# Add the total row to the DataFrame
agg.loc["total"] = total_row

# Sort the columns
agg = agg.sort_index(axis=1)

agg.reset_index()

ds_name all_complete found_cell_type_id                      \
                                                complete   found         pct   
0      adipose_Emont2022        False               True   373.0  100.000000   
1   adipose_Hildreth2021        False              False   162.0   95.294118   
2     adipose_Jaitin2019         True               True    11.0  100.000000   
3    adipose_Massier2023        False               True   258.0  100.000000   
4    adipose_Merrick2019        False              False    28.0   96.551724   
5      adipose_Vijay2019        False               True   105.0  100.000000   
6         bladder_Yu2019        False               True    79.0  100.000000   
7            bone_He2021        False              False   398.0   99.749373   
8         eye_Gautam2021        False               True   373.0  100.000000   
9       heart_Tucker2020        False               True   559.0  100.000000   
10                 total        False              False  2346.0   99.575552   

           found_feature_id                              
     total         complete   found         pct   total  
0    373.0            False   367.0   98.391421   373.0  
1    170.0            False   160.0   94.117647   170.0  
2     11.0             True    11.0  100.000000    11.0  
3    258.0            False   251.0   97.286822   258.0  
4     29.0            False    21.0   72.413793    29.0  
5    105.0            False   103.0   98.095238   105.0  
6     79.0            False    75.0   94.936709    79.0  
7    399.0            False   385.0   96.491228   399.0  
8    373.0            False   352.0   94.369973   373.0  
9    559.0            False   531.0   94.991055   559.0  
10  2356.0            False  2256.0   95.755518  2356.0

In [141]:
# find specific datasets with missing cell type or feature name
ds = 'bone_He2021'
llm.query(
    f"ds_name == '{ds}' & (~found_cell_type_id | ~found_feature_id)"
)[["cell_type_label", "cell_type_id",  "cell_type_id_mapped", "feature_name", "feature_identifier",  "feature_id_mapped"]].head(3)

,cell_type_label,cell_type_id,cell_type_id_mapped,feature_name,feature_identifier,feature_id_mapped
15,PERICHONDRIAL MESENCHYMAL STROMAL CELLS (PMSCS),PERICHONDRIAL MESENCHYMAL STROMAL CELL,PERICHONDRIAL MESENCHYMAL STROMAL CELL,NOV,ENSG00000147403,ENSG00000136999
20,NEURON,NEURON,NEURON,HOX2,ENSG00000170689,ENSG00000120094
21,NEURON,NEURON,NEURON,HOX3,ENSG00000180806,ENSG00000123407
